# 5 · Deploy your own dashboards

You built the tables. This notebook puts a front end on them.

Two Streamlit apps, deployed **from the Git repository** straight into your own
schema. No file uploads, no stage to manage: Snowflake reads the code from Git.

| App | Rights | What it shows |
| --- | --- | --- |
| Internal MI | owner's | the whole market, as an internal analyst sees it |
| Partner Insights | **caller's** | only what the *viewer's* role is allowed to see |

That second one is the interesting one. It has **no filtering logic in it at
all**. The row access policy you built in notebook 04 decides what exists, so
`ZIXTY_USER` sees one insurer and you see all seven — from identical code.

Both run on **container runtime**, for one reason: a Cortex Agent cannot be
called from a warehouse-runtime Streamlit app.

## Before you start

Notebooks 01, 03 and 04 must have run — the apps read your gold tables, the
agent, and your partner role.

The facilitator must also have created, once for the whole account:

- the API integration and Git repository (`DEFAQTO_DB.PUBLIC.WORKSHOP_REPO`)
- a compute pool with room for everyone's apps
- `GRANT DATABASE ROLE SNOWFLAKE.PYPI_REPOSITORY_USER TO ROLE ACCOUNTADMIN`

All three are in `setup/00_admin_setup.sql`. The next cell checks them rather
than assuming, because each one fails in a way that does not obviously point at
the cause.

In [ ]:
-- >>> THE TWO LINES YOU EDIT <<<
-- The same alias and insurer you used in notebooks 01 and 04.
SET alias      = 'CHANGEME';
SET my_insurer = 'ZIXTY';          -- ZIXTY | VEYGO | COVERTIME | SAFELYINSURED

-- Session variables do NOT carry over from notebook 04, so my_role is rebuilt
-- here from the same rule that notebook created it with.
SET my_schema = (SELECT 'DEFAQTO_DB.TRANSFORMED_' || UPPER($alias));
SET up        = (SELECT UPPER($alias));
SET my_role   = (SELECT 'PARTNER_' || REPLACE(UPPER($my_insurer), ' ', '_'));
SET pool      = 'DEFAQTO_HOL_POOL';        -- ask the facilitator if unsure

USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA IDENTIFIER($my_schema);

SELECT $my_schema AS deploying_into, $up AS name_suffix,
       $my_role AS partner_role, $pool AS compute_pool;

## 1 · Preflight

Four things, each of which has broken this before:

- **The gold tables exist.** The apps query all three.
- **The Git repository is fetched.** `FETCH` is not automatic — Snowflake keeps
  serving the commit it last saw, so a fix pushed to GitHub can appear to have
  no effect.
- **The compute pool has capacity.** Every container app is one SPCS service.
  `SYSTEM_COMPUTE_POOL_CPU` is capped at 2 nodes and cannot be resized, which is
  why the workshop uses its own pool.
- **The PyPI grant.** Without it the app fails to *build*, reporting an
  artifact-repository error rather than anything permission-shaped.

In [ ]:
-- Gold tables (expect 3).
SHOW DYNAMIC TABLES LIKE 'GOLD%' IN SCHEMA IDENTIFIER($my_schema);

-- Pull the latest commit. Run this again after any push to GitHub.
ALTER GIT REPOSITORY DEFAQTO_DB.PUBLIC.WORKSHOP_REPO FETCH;

-- The app source, as Snowflake now sees it (expect 4 files).
LIST '@DEFAQTO_DB.PUBLIC.WORKSHOP_REPO/branches/main/streamlit/';

-- Capacity, and the PyPI grant.
SHOW COMPUTE POOLS;
SHOW GRANTS TO ROLE ACCOUNTADMIN;

## 2 · The internal dashboard

`FROM` takes a **literal**, so it cannot be concatenated — but the Git path is
the same for everyone, and only the app's name and schema differ. So the
statement is built as a string and executed.

The name carries your alias. Not to avoid a clash — different schemas already
make these different objects — but because every attendee here is
`ACCOUNTADMIN`, so **you will all see all of the apps** in Snowsight. Without
the suffix you would be looking at three identically titled entries.

In [ ]:
SET st = (SELECT REPLACE(REPLACE($$
CREATE OR REPLACE STREAMLIT @@SCHEMA@@.DEFAQTO_INTERNAL_MI_@@ALIAS@@
  FROM '@DEFAQTO_DB.PUBLIC.WORKSHOP_REPO/branches/main/streamlit/internal_mi/'
  MAIN_FILE             = 'streamlit_app.py'
  QUERY_WAREHOUSE       = COMPUTE_WH
  RUNTIME_NAME          = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
  COMPUTE_POOL          = @@POOL@@
  ARTIFACT_REPOSITORIES = (snowflake.snowpark.pypi_shared_repository)
  TITLE                 = 'Defaqto Internal MI (@@ALIAS@@)'
  COMMENT               = 'Whole-market view. Owner rights.'
$$, '@@SCHEMA@@', $my_schema), '@@ALIAS@@', $up));

SET st = (SELECT REPLACE($st, '@@POOL@@', $pool));
EXECUTE IMMEDIATE $st;

-- An app is NOT live until a live version exists. Skip this and it works for
-- you and for nobody else.
SET live = (SELECT 'ALTER STREAMLIT ' || $my_schema || '.DEFAQTO_INTERNAL_MI_' || $up
                   || ' ADD LIVE VERSION FROM LAST');
EXECUTE IMMEDIATE $live;

## 3 · The partner dashboard

Same code, one difference that matters: this app connects with
`st.connection("snowflake-callers-rights")`, so queries run as **the viewer**,
not as the app's owner. That is what makes the row access policy do the work.

Under owner's rights the policy would evaluate `ACCOUNTADMIN` — which it exempts
— and a partner would see all seven insurers. The app would look like it worked
and be completely wrong.

In [ ]:
SET st = (SELECT REPLACE(REPLACE($$
CREATE OR REPLACE STREAMLIT @@SCHEMA@@.DEFAQTO_PARTNER_INSIGHTS_@@ALIAS@@
  FROM '@DEFAQTO_DB.PUBLIC.WORKSHOP_REPO/branches/main/streamlit/partner_insights/'
  MAIN_FILE             = 'streamlit_app.py'
  QUERY_WAREHOUSE       = COMPUTE_WH
  RUNTIME_NAME          = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
  COMPUTE_POOL          = @@POOL@@
  ARTIFACT_REPOSITORIES = (snowflake.snowpark.pypi_shared_repository)
  TITLE                 = 'Defaqto Partner Insights (@@ALIAS@@)'
  COMMENT               = 'Partner view. Caller rights - the policy decides what exists.'
$$, '@@SCHEMA@@', $my_schema), '@@ALIAS@@', $up));

SET st = (SELECT REPLACE($st, '@@POOL@@', $pool));
EXECUTE IMMEDIATE $st;

SET live = (SELECT 'ALTER STREAMLIT ' || $my_schema || '.DEFAQTO_PARTNER_INSIGHTS_' || $up
                   || ' ADD LIVE VERSION FROM LAST');
EXECUTE IMMEDIATE $live;

## 4 · Caller grants

Caller's rights needs permission from **both** directions, which is the part
that catches people out.

The viewer's own role needs `SELECT` — your partner role got that in notebook
04. Separately, the role that **owns the app** needs a *caller* privilege on
each object, or Snowflake refuses to reach through at all:

> `This executable runs with restricted caller's rights. The owner role
> SYSTEM$MANAGED must have at least one CALLER privilege granted on TABLE …`

Note the pre-computed variables. `IDENTIFIER()` accepts a plain session variable
but **not** a concatenated expression inside a `GRANT` — that fails with
`syntax error … unexpected '||'`.

In [ ]:
SET cg_prov = $my_schema || '.GOLD_PROVIDER_DAILY';
SET cg_coh  = $my_schema || '.GOLD_COHORT_CONVERSION';
SET cg_fun  = $my_schema || '.GOLD_FUNNEL_DAILY';

GRANT CALLER USAGE  ON DATABASE DEFAQTO_DB              TO ROLE ACCOUNTADMIN;
GRANT CALLER USAGE  ON SCHEMA   IDENTIFIER($my_schema)  TO ROLE ACCOUNTADMIN;

GRANT CALLER SELECT ON DYNAMIC TABLE IDENTIFIER($cg_prov) TO ROLE ACCOUNTADMIN;
GRANT CALLER SELECT ON DYNAMIC TABLE IDENTIFIER($cg_coh)  TO ROLE ACCOUNTADMIN;
GRANT CALLER SELECT ON DYNAMIC TABLE IDENTIFIER($cg_fun)  TO ROLE ACCOUNTADMIN;

SHOW CALLER GRANTS TO ROLE ACCOUNTADMIN;

## 5 · Let your partner in

A container-runtime app needs **three** grants per viewer role, not one: the
Streamlit, the service behind it, and the service's `STREAMLIT_VIEWER` role.
Service names are generated, so they are read back rather than typed.

Run the `SHOW SERVICES` below, find the service whose name matches your app,
then fill it into the two commented lines.

In [ ]:
-- Which service belongs to your app?
SHOW SERVICES IN COMPUTE POOL IDENTIFIER($pool);

-- The Streamlit itself.
SET g = (SELECT 'GRANT USAGE ON STREAMLIT ' || $my_schema
                || '.DEFAQTO_PARTNER_INSIGHTS_' || $up
                || ' TO ROLE ' || $my_role);
EXECUTE IMMEDIATE $g;

-- Then the service and its viewer role. Substitute the service name from above:
--
--   GRANT USAGE ON SERVICE      <db>.<schema>.<service>                   TO ROLE <your PARTNER_ role>;
--   GRANT USAGE ON SERVICE ROLE <db>.<schema>.<service>.STREAMLIT_VIEWER  TO ROLE <your PARTNER_ role>;

SHOW GRANTS TO ROLE IDENTIFIER($my_role);

## 6 · Prove it

Open both apps from **Projects » Streamlit**, then sign in as your partner user
and open the partner app again.

| Signed in as | Insurers visible |
| --- | --- |
| you (`ACCOUNTADMIN`) | 7 |
| your partner user | **1** |

Same app, same code, same URL. If the partner sees 7, work through these in
order:

1. **The partner user's DEFAULT role.** Caller's rights uses the *default* role,
   not whatever is selected in Snowsight. If it is `ACCOUNTADMIN`, the policy
   exempts it.
2. **Is the policy still attached?** `CREATE OR REPLACE DYNAMIC TABLE` detaches
   row access policies silently. Re-run notebook 04 from `attach_policy`.
3. **Cache.** On container runtime every viewer shares one app instance, and
   `st.cache_data` is global to it. These apps include `CURRENT_ROLE()` in the
   cache key for exactly this reason — if you edit them, keep it.

In [ ]:
-- As your partner role.
USE ROLE IDENTIFIER($my_role);
SELECT CURRENT_ROLE()               AS acting_as,
       COUNT(DISTINCT PROVIDER_KEY) AS insurers_visible,
       COUNT(*)                     AS rows_visible
FROM   IDENTIFIER($cg_prov);

-- As yourself. Nothing about the query changed - only who is asking.
USE ROLE ACCOUNTADMIN;
SELECT CURRENT_ROLE()               AS acting_as,
       COUNT(DISTINCT PROVIDER_KEY) AS insurers_visible,
       COUNT(*)                     AS rows_visible
FROM   IDENTIFIER($cg_prov);

## 7 · After a change

`FROM` copies the files **once**. A later `git push` does not update a deployed
app.

Do **not** fix it with `CREATE OR REPLACE` — that drops the live version and
every grant, including the caller grants, and the app then fails for everyone
but you. Add a version instead:

In [ ]:
ALTER GIT REPOSITORY DEFAQTO_DB.PUBLIC.WORKSHOP_REPO FETCH;

SET v = (SELECT 'ALTER STREAMLIT ' || $my_schema || '.DEFAQTO_PARTNER_INSIGHTS_' || $up
                || ' ADD VERSION FROM ''@DEFAQTO_DB.PUBLIC.WORKSHOP_REPO/branches/main/streamlit/partner_insights/''');
EXECUTE IMMEDIATE $v;

SET v = (SELECT 'ALTER STREAMLIT ' || $my_schema || '.DEFAQTO_PARTNER_INSIGHTS_' || $up
                || ' ADD LIVE VERSION FROM LAST');
EXECUTE IMMEDIATE $v;

-- All viewers share one container, so the new code lands when that container
-- restarts. If the old behaviour persists, suspend and resume the pool.

## 8 · Removing just the apps

Frees the compute pool without touching anything else you built.

`ARTIFACT_REPOSITORIES` is worth knowing about here: it appears in
`SHOW STREAMLITS` but reads as `None` in `DESCRIBE STREAMLIT`, so check with
`SHOW` or you will conclude it was never attached.

In [ ]:
SET d = (SELECT 'DROP STREAMLIT IF EXISTS ' || $my_schema || '.DEFAQTO_INTERNAL_MI_' || $up);
EXECUTE IMMEDIATE $d;

SET d = (SELECT 'DROP STREAMLIT IF EXISTS ' || $my_schema || '.DEFAQTO_PARTNER_INSIGHTS_' || $up);
EXECUTE IMMEDIATE $d;

SHOW STREAMLITS IN SCHEMA IDENTIFIER($my_schema);